# Mikey — Final Kaggle 14B Private AI

A branded private AI chatbot running on Kaggle with llama.cpp, Flask, and a temporary Cloudflare Quick Tunnel.

**Flow:** Browser → Cloudflare → Flask → llama.cpp → Qwen3 14B GGUF → Kaggle GPU

Run every cell from top to bottom in a fresh Kaggle GPU session.


In [3]:
# CELL 1 — GPU / environment check

import subprocess
import sys

print("=" * 70)
print("GPU CHECK")
print("=" * 70)
subprocess.run(["nvidia-smi"], check=False)

print("\nPython:", sys.version.split()[0])

try:
    r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    print("\nCUDA:")
    print(r.stdout.strip())
except FileNotFoundError:
    print("\nnvcc not found — compilation is not required.")


GPU CHECK
Sat Sep 12 18:20:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------------

In [4]:
# CELL 2 — Kaggle dependency check
# No pip installation unless a package is genuinely missing.

import sys
import importlib

required = {
    "requests": "requests",
    "flask": "flask",
    "huggingface_hub": "huggingface_hub",
}

missing = []

print("=" * 70)
print("CHECKING PYTHON DEPENDENCIES")
print("=" * 70)

for module, package in required.items():
    try:
        imported = importlib.import_module(module)
        version = getattr(imported, "__version__", "installed")
        print(f"✓ {package}: {version}")
    except ImportError:
        print(f"✗ {package}: MISSING")
        missing.append(package)

# ------------------------------------------------------------
# Only try pip if something is actually missing
# ------------------------------------------------------------

if missing:
    print("\nMissing packages:", ", ".join(missing))
    print("Attempting installation...")

    import subprocess

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-input",
        *missing,
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        print("\nPIP ERROR:")
        print(result.stderr)

        raise RuntimeError(
            "Kaggle could not install the missing packages. "
            "The notebook needs Internet enabled."
        )

    print("Installation completed.")

    # Verify again
    for module, package in required.items():
        try:
            imported = importlib.import_module(module)
            print(f"✓ {package}: OK")
        except ImportError:
            raise RuntimeError(
                f"{package} is still unavailable after installation."
            )

print("\n" + "=" * 70)
print("DEPENDENCIES READY")
print("=" * 70)

CHECKING PYTHON DEPENDENCIES
✓ requests: 2.34.2
✓ flask: 3.1.3
✓ huggingface_hub: 1.31.0

DEPENDENCIES READY


/tmp/ipykernel_8133/135064550.py:22: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  version = getattr(imported, "__version__", "installed")


In [12]:
# CELL 3 — Configuration / customization

from pathlib import Path

AI_NAME = "Mikey"
WEBSITE_NAME = "codingcat"
WEBSITE_TAGLINE = "A private AI chatbot."

MODEL_REPO = "bartowski/huihui-ai_Qwen3-14B-abliterated-GGUF"
MODEL_FILE = "huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf"
MODEL_DIR = Path("/kaggle/working/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / MODEL_FILE

LLAMA_PORT = 8080
WEB_PORT = 5000

CONTEXT_SIZE = 8192
TEMPERATURE = 0.8
TOP_P = 0.95
MAX_TOKENS = 2000

print("AI:", AI_NAME)
print("Website:", WEBSITE_NAME)
print("Model:", MODEL_FILE)
print("Path:", MODEL_PATH)


AI: Mikey
Website: codingcat
Model: huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf
Path: /kaggle/working/models/huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf


In [4]:
# CELL 4 — Download the 14B GGUF only if missing

from huggingface_hub import hf_hub_download

if MODEL_PATH.exists():
    print("Existing 14B model found — skipping download.")
else:
    print("14B model not found. Downloading...")
    MODEL_PATH = Path(
        hf_hub_download(
            repo_id=MODEL_REPO,
            filename=MODEL_FILE,
            local_dir=str(MODEL_DIR),
        )
    )
    print("Download complete.")

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model missing: {MODEL_PATH}")

print("Path:", MODEL_PATH)
print(f"Size: {MODEL_PATH.stat().st_size / (1024**3):.2f} GB")


Existing 14B model found — skipping download.
Path: /kaggle/working/models/huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf
Size: 8.38 GB


In [9]:
# CELL 5 — Download a prebuilt llama.cpp CUDA binary for Tesla T4 (SM75)

from pathlib import Path
import requests
import tarfile
import shutil
import os

INSTALL_DIR = Path("/kaggle/working/llama_cpp")
INSTALL_DIR.mkdir(parents=True, exist_ok=True)

# Tesla T4 = compute capability 7.5
SM = "75"

print("Searching for a prebuilt llama.cpp CUDA binary...")
print("GPU target: NVIDIA Tesla T4 (SM75)")

# Ask GitHub for recent releases
api_url = "https://api.github.com/repos/cloudlnkcn/llama.cpp/releases"

r = requests.get(
    api_url,
    headers={"Accept": "application/vnd.github+json"},
    params={"per_page": 20},
    timeout=30,
)

r.raise_for_status()
releases = r.json()

asset = None
release_tag = None

# Find a CUDA SM75 Linux asset
for release in releases:
    for a in release.get("assets", []):
        name = a.get("name", "")

        if (
            "ubuntu-cuda-sm_75-x64.tar.xz" in name
            and name.endswith(".tar.xz")
        ):
            asset = a
            release_tag = release.get("tag_name")
            break

    if asset:
        break

if asset is None:
    print("Could not find the exact SM75 binary.")
    print("Available CUDA assets found:")

    for release in releases:
        for a in release.get("assets", []):
            name = a.get("name", "")
            if "ubuntu-cuda" in name.lower():
                print(" ", release.get("tag_name"), name)

    raise RuntimeError(
        "No prebuilt Linux CUDA SM75 llama.cpp binary was found."
    )

asset_url = asset["browser_download_url"]
asset_name = asset["name"]

print(f"Found release: {release_tag}")
print(f"Asset: {asset_name}")
print("Downloading...")

archive_path = Path("/kaggle/working") / asset_name

with requests.get(asset_url, stream=True, timeout=60) as response:
    response.raise_for_status()

    total = int(response.headers.get("content-length", 0))
    downloaded = 0

    with open(archive_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

                if total:
                    pct = downloaded * 100 / total
                    print(
                        f"\rProgress: {pct:6.2f}%",
                        end="",
                        flush=True
                    )

print("\nDownload complete.")

# Extract
print("Extracting...")

with tarfile.open(archive_path, "r:xz") as tar:
    tar.extractall(INSTALL_DIR)

# Find llama-server
server_candidates = list(INSTALL_DIR.rglob("llama-server"))

if not server_candidates:
    raise FileNotFoundError(
        "llama-server was not found after extracting the CUDA package."
    )

LLAMA_SERVER = str(server_candidates[0])

# Make executable
os.chmod(LLAMA_SERVER, 0o755)

print()
print("✅ llama.cpp CUDA server ready")
print("Server:", LLAMA_SERVER)

# Remove archive to save disk space
archive_path.unlink(missing_ok=True)

print("Archive removed to save disk space.")

Searching for a prebuilt llama.cpp CUDA binary...
GPU target: NVIDIA Tesla T4 (SM75)
Found release: v0.4.0
Asset: llama-v0.4.0-ubuntu-cuda-sm_75-x64.tar.xz
Downloading...
Progress: 100.00%
Download complete.
Extracting...


/tmp/ipykernel_8133/2448120059.py:99: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(INSTALL_DIR)



✅ llama.cpp CUDA server ready
Server: /kaggle/working/llama_cpp/llama-b10809/llama-server
Archive removed to save disk space.


In [10]:
# CELL 6 — Verify both Tesla T4 GPUs

result = subprocess.run(
    [LLAMA_SERVER, "--list-devices"],
    capture_output=True,
    text=True,
    env=os.environ.copy(),
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("CUDA devices could not be enumerated.")


Available devices:
  CUDA0: Tesla T4 (14911 MiB, 14806 MiB free)
  CUDA1: Tesla T4 (14911 MiB, 14806 MiB free)



In [15]:
# CELL 7 — Start llama-server

import os
import subprocess
import time
import requests

# Kill any old llama-server processes
subprocess.run(
    ["pkill", "-f", "llama-server"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(2)

# Make sure variables are strings
LLAMA_SERVER = str(LLAMA_SERVER)
MODEL_PATH = str(MODEL_PATH)

print("llama-server:", LLAMA_SERVER)
print("Model:", MODEL_PATH)

# Verify files actually exist
if not os.path.isfile(LLAMA_SERVER):
    raise FileNotFoundError(
        f"llama-server binary not found:\n{LLAMA_SERVER}"
    )

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found:\n{MODEL_PATH}"
    )

os.chmod(LLAMA_SERVER, 0o755)

server_cmd = [
    LLAMA_SERVER,

    "-m", MODEL_PATH,

    "--host", "127.0.0.1",
    "--port", "8080",

    # Two Tesla T4 GPUs
    "--split-mode", "layer",
    "--tensor-split", "1,1",
    "--n-gpu-layers", "all",

    "--ctx-size", "8192",

    "--temp", "0.8",
    "--top-p", "0.95",

    "--parallel", "1",
]

print("\nStarting llama-server...")
print(" ".join(server_cmd))

server_log = open(
    "/tmp/llama-server.log",
    "w"
)

llama_process = subprocess.Popen(
    server_cmd,
    stdout=server_log,
    stderr=subprocess.STDOUT,
    text=True
)

# Give it some time to initialize
print("\nWaiting for llama-server...")

for i in range(60):

    time.sleep(2)

    if llama_process.poll() is not None:

        print("\n❌ llama-server EXITED")
        print("Exit code:", llama_process.returncode)

        server_log.close()

        print("\n========== SERVER LOG ==========\n")

        with open("/tmp/llama-server.log") as f:
            print(f.read()[-12000:])

        raise RuntimeError(
            "llama-server failed to start."
        )

    try:

        r = requests.get(
            "http://127.0.0.1:8080/health",
            timeout=2
        )

        if r.status_code == 200:

            print("\n✅ llama-server is READY")
            print(r.text)

            break

    except Exception:
        pass

else:

    print("\n❌ llama-server did not become ready.")

    server_log.close()

    print("\n========== SERVER LOG ==========\n")

    with open("/tmp/llama-server.log") as f:
        print(f.read()[-12000:])

    raise RuntimeError(
        "llama-server did not start within 120 seconds."
    )

llama-server: /kaggle/working/llama_cpp/llama-b10809/llama-server
Model: /kaggle/working/models/huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf

Starting llama-server...
/kaggle/working/llama_cpp/llama-b10809/llama-server -m /kaggle/working/models/huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf --host 127.0.0.1 --port 8080 --split-mode layer --tensor-split 1,1 --n-gpu-layers all --ctx-size 8192 --temp 0.8 --top-p 0.95 --parallel 1

Waiting for llama-server...

✅ llama-server is READY
{"status":"ok"}


In [16]:
# CELL 9 — Test llama-server API

import requests
import time

LLAMA_URL = f"http://127.0.0.1:{LLAMA_PORT}"

print("Testing llama-server...")
print("URL:", LLAMA_URL)

# Give the server a moment
time.sleep(2)

# Health check
try:
    r = requests.get(
        f"{LLAMA_URL}/health",
        timeout=10
    )

    print("Health status:", r.status_code)
    print("Health response:", r.text)

except Exception as e:

    raise RuntimeError(
        f"llama-server is not reachable:\n{e}"
    )


# Actual chat test
payload = {
    "messages": [
        {
            "role": "user",
            "content": "Say hello in one short sentence."
        }
    ],
    "temperature": 0.7,
    "max_tokens": 50,
    "stream": False
}

print("\nSending test prompt...")

try:

    r = requests.post(
        f"{LLAMA_URL}/v1/chat/completions",
        json=payload,
        timeout=180
    )

    r.raise_for_status()

    data = r.json()

    answer = (
        data
        .get("choices", [{}])[0]
        .get("message", {})
        .get("content", "")
    )

    print("\n======================================")
    print("✅ MODEL TEST SUCCESSFUL")
    print("======================================")
    print("MyAI:", answer)

except Exception as e:

    print("\n❌ Chat request failed")
    print(e)

    if "r" in locals():
        print("\nServer response:")
        print(r.text[:5000])

    raise

Testing llama-server...
URL: http://127.0.0.1:8080
Health status: 200
Health response: {"status":"ok"}

Sending test prompt...

✅ MODEL TEST SUCCESSFUL
MyAI: <think>
Okay, the user wants me to say hello in one short sentence. Let me think. They probably want a simple and friendly greeting. Maybe something like "Hello! How are you today?" But wait, that's two sentences. Let me


In [17]:
# CELL 11 — START PUBLIC CLOUDFLARE QUICK TUNNEL
# Flask MUST be running on port 5000 before this cell.

import os
import re
import time
import subprocess
import urllib.request

CLOUDFLARED = "/kaggle/working/cloudflared"
WEB_PORT = 5000
LOG_PATH = "/tmp/mikey_cloudflared.log"

# Download cloudflared if necessary.
if not os.path.exists(CLOUDFLARED):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CLOUDFLARED,
    )
    os.chmod(CLOUDFLARED, 0o755)

print(subprocess.run(
    [CLOUDFLARED, "--version"],
    capture_output=True,
    text=True,
).stdout.strip())

# Kill old tunnel only.
subprocess.run(
    ["bash","-lc","pkill -x cloudflared 2>/dev/null || true"],
    capture_output=True,
)
time.sleep(1)

# Start a fresh Quick Tunnel against the CORRECT Flask port.
log = open(LOG_PATH, "w")

cloudflare_process = subprocess.Popen(
    [
        CLOUDFLARED,
        "tunnel",
        "--url", f"http://127.0.0.1:{WEB_PORT}",
        "--protocol", "http2",
        "--edge-ip-version", "4",
        "--no-autoupdate",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None

for _ in range(30):
    time.sleep(1)
    try:
        content = open(LOG_PATH, "r").read()
        m = re.search(
            r"https://[A-Za-z0-9-]+\.trycloudflare\.com",
            content,
        )
        if m:
            public_url = m.group(0)
            break
    except Exception:
        pass

print("=" * 70)

if public_url:
    print("🔥 MIKEY IS LIVE")
    print("=" * 70)
    print(public_url)
    print("=" * 70)
    print("Backend: llama-server :8080")
    print("Web UI:  Flask :5000")
    print("=" * 70)
else:
    print("❌ Cloudflare did not produce a public URL.")
    print("---- cloudflared log ----")
    print(open(LOG_PATH).read()[-10000:])
    raise RuntimeError("Cloudflare Quick Tunnel failed.")


cloudflared version 2026.9.1 (built 2026-09-11-13:35 UTC)
🔥 MIKEY IS LIVE
https://meets-scope-casting-bases.trycloudflare.com
Backend: llama-server :8080
Web UI:  Flask :5000


In [18]:
# ============================================================
# MIKEY — WINDOWS CMD STYLE UI
# ============================================================

import os
import sys
import time
import subprocess
import threading
import requests

from flask import Flask, request, jsonify, Response


# ============================================================
# CONFIG
# ============================================================

AI_NAME = "Mikey"
WEB_PORT = 5000
LLAMA_URL = f"http://127.0.0.1:{LLAMA_PORT}"


# ============================================================
# CLEAN OLD FLASK
# ============================================================

os.makedirs("/tmp/mikey_cmd", exist_ok=True)
os.chdir("/tmp/mikey_cmd")

print("Starting Mikey CMD interface...")

# Kill whatever is currently using port 5000
subprocess.run(
    ["bash", "-lc", "fuser -k 5000/tcp 2>/dev/null || true"],
    capture_output=True
)

time.sleep(1)


# ============================================================
# FLASK
# ============================================================

app = Flask("mikey_cmd")


# ============================================================
# CMD UI
# ============================================================

@app.route("/")
def home():

    html = r"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>__AI_NAME__</title>


<style>

/* ============================================================
   WINDOWS CMD LOOK
   ============================================================ */

html,
body {

    margin: 0;
    padding: 0;

    width: 100%;
    height: 100%;

    background: #0c0c0c;

    color: #cccccc;

    font-family:
        "Cascadia Mono",
        "Cascadia Code",
        Consolas,
        "Lucida Console",
        "Courier New",
        monospace;

    font-size: 16px;

    font-weight: 400;

}


/* ============================================================
   PAGE
   ============================================================ */

body {

    overflow: hidden;

}


/* ============================================================
   CMD AREA
   ============================================================ */

#cmd {

    position: fixed;

    top: 0;
    left: 0;
    right: 0;
    bottom: 0;

    padding: 5px 6px;

    background: #0c0c0c;

    color: #cccccc;

    overflow-y: auto;
    overflow-x: hidden;

    white-space: pre-wrap;

    word-break: break-word;

    line-height: 1.25;

    box-sizing: border-box;

}


/* ============================================================
   EACH LINE
   ============================================================ */

.line {

    margin: 0;
    padding: 0;

    min-height: 20px;

}


/* ============================================================
   USER COMMAND
   ============================================================ */

.user {

    color: #cccccc;

}


/* ============================================================
   AI OUTPUT
   ============================================================ */

.ai {

    color: #cccccc;

}


/* ============================================================
   INPUT LINE
   ============================================================ */

#inputLine {

    display: flex;

    width: 100%;

    margin: 0;
    padding: 0;

    align-items: flex-start;

}


#prompt {

    flex: none;

    color: #cccccc;

    white-space: pre;

}


#input {

    flex: 1;

    width: 100%;

    min-width: 0;

    margin: 0;
    padding: 0;

    border: none;
    outline: none;

    background: transparent;

    color: #cccccc;

    font-family:
        "Cascadia Mono",
        "Cascadia Code",
        Consolas,
        "Lucida Console",
        "Courier New",
        monospace;

    font-size: 16px;

    font-weight: 400;

    line-height: 1.25;

    resize: none;

    overflow: hidden;

    caret-color: #cccccc;

}


#input::selection {

    background: #cccccc;

    color: #0c0c0c;

}


/* ============================================================
   CMD-STYLE SCROLLBAR
   ============================================================ */

#cmd::-webkit-scrollbar {

    width: 12px;

}


#cmd::-webkit-scrollbar-track {

    background: #0c0c0c;

}


#cmd::-webkit-scrollbar-thumb {

    background: #3f3f3f;

}


#cmd::-webkit-scrollbar-thumb:hover {

    background: #5a5a5a;

}


/* Firefox */

#cmd {

    scrollbar-color: #3f3f3f #0c0c0c;

    scrollbar-width: auto;

}

</style>

</head>


<body>


<div id="cmd">


<div id="history"></div>


<div id="inputLine">

    <span id="prompt">You&gt; </span>

    <textarea
        id="input"
        rows="1"
        autocomplete="off"
        spellcheck="false"
        autofocus
    ></textarea>

</div>


</div>


<script>

/* ============================================================
   VARIABLES
   ============================================================ */

const NAME = "__AI_NAME__";

const cmd =
    document.getElementById("cmd");

const history =
    document.getElementById("history");

const input =
    document.getElementById("input");


let busy = false;


/* ============================================================
   SCROLL TO BOTTOM
   ============================================================ */

function scrollBottom() {

    cmd.scrollTop =
        cmd.scrollHeight;

}


/* ============================================================
   ADD CMD LINE
   ============================================================ */

function addLine(text, type="user") {

    const line =
        document.createElement("div");

    line.className =
        "line " + type;

    line.textContent =
        text;

    history.appendChild(line);

    scrollBottom();

    return line;

}


/* ============================================================
   REMOVE THINKING TAGS
   ============================================================ */

function cleanResponse(text) {

    if (!text)
        return "";

    text =
        text.replace(
            /<think>[\s\S]*?<\/think>/gi,
            ""
        );

    text =
        text.replace(
            /<think>[\s\S]*$/gi,
            ""
        );

    return text.trim();

}


/* ============================================================
   INPUT RESIZE
   ============================================================ */

function resizeInput() {

    input.style.height = "20px";

    input.style.height =
        Math.min(
            input.scrollHeight,
            180
        ) + "px";

}


input.addEventListener(
    "input",
    resizeInput
);


/* ============================================================
   SEND
   ============================================================ */

async function sendMessage() {

    if (busy)
        return;


    const message =
        input.value.trim();


    if (!message)
        return;


    busy = true;


    /*
       Turn the current command into a
       permanent CMD line.
    */

    addLine(
        "You> " + message,
        "user"
    );


    input.value = "";

    resizeInput();


    /*
       Temporary response line.
    */

    const responseLine =
        addLine(
            NAME + "> ...Thinking",
            "ai"
        );


    try {

        const response =
            await fetch(
                "/chat",
                {
                    method: "POST",

                    headers: {
                        "Content-Type":
                            "application/json"
                    },

                    body: JSON.stringify({
                        message: message
                    })
                }
            );


        const data =
            await response.json();


        responseLine.remove();


        if (!response.ok) {

            addLine(
                NAME +
                "> ERROR: " +
                (
                    data.error ||
                    "Request failed."
                ),
                "ai"
            );

        }

        else {

            const answer =
                cleanResponse(
                    data.response || ""
                );


            /*
               Preserve newlines exactly like
               command-line output.
            */

            const parts =
                answer.split("\n");


            parts.forEach(
                function(part, index) {

                    if (index === 0) {

                        addLine(
                            NAME +
                            "> " +
                            part,
                            "ai"
                        );

                    }

                    else {

                        addLine(
                            part,
                            "ai"
                        );

                    }

                }
            );

        }

    }

    catch (error) {

        responseLine.remove();

        addLine(
            NAME +
            "> ERROR: " +
            error.message,
            "ai"
        );

    }


    busy = false;

    input.focus();

    scrollBottom();

}


/* ============================================================
   ENTER = EXECUTE
   SHIFT + ENTER = NEW LINE
   ============================================================ */

input.addEventListener(
    "keydown",
    function(event) {

        if (
            event.key === "Enter" &&
            !event.shiftKey
        ) {

            event.preventDefault();

            sendMessage();

        }

    }
);


/* ============================================================
   CLICK ANYWHERE = FOCUS
   ============================================================ */

document.addEventListener(
    "click",
    function() {

        input.focus();

    }
);


input.focus();

resizeInput();

scrollBottom();

</script>


</body>

</html>
"""

    html = html.replace(
        "__AI_NAME__",
        AI_NAME
    )

    return Response(
        html,
        mimetype="text/html",

        headers={
            "Cache-Control":
                "no-store, no-cache, must-revalidate, max-age=0",

            "Pragma":
                "no-cache",

            "Expires":
                "0"
        }
    )


# ============================================================
# HEALTH
# ============================================================

@app.route("/health")
def health():

    return jsonify({
        "status": "ok",
        "service": AI_NAME
    })


# ============================================================
# CHAT
# ============================================================

@app.route("/chat", methods=["POST"])
def chat():

    data =request.get_json(
            silent=True
        ) or {}

    message = data.get(
            "message",
            ""
        ).strip()


    if not message:

        return jsonify({
            "error": "Empty message"
        }), 400


    payload = {

        "messages": [
            {
                "role": "user",
                "content": message
            }
        ],

        "temperature":
            TEMPERATURE,

        "top_p":
            TOP_P,

        "max_tokens":
            MAX_TOKENS,

        "stream":
            False
    }


    try:

        response =requests.post(
                f"{LLAMA_URL}/v1/chat/completions",

                json=payload,

                timeout=300
            )


        response.raise_for_status()


        result =response.json()


        answer = (
            result
            .get(
                "choices",
                [{}]
            )[0]
            .get(
                "message",
                {}
            )
            .get(
                "content",
                ""
            )
        )


        return jsonify({
            "response": answer
        })


    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 500


# ============================================================
# START
# ============================================================

def run_flask():

    app.run(
        host="0.0.0.0",

        port=WEB_PORT,

        debug=False,

        use_reloader=False,

        threaded=True
    )


web_thread =threading.Thread(
        target=run_flask,
        daemon=True
    )


web_thread.start()


# ============================================================
# VERIFY
# ============================================================

time.sleep(3)


try:

    health =requests.get(
            f"http://127.0.0.1:{WEB_PORT}/health",
            timeout=10
        )


    page = requests.get(
            f"http://127.0.0.1:{WEB_PORT}/",
            timeout=10
        )


    print()
    print("=" * 60)
    print("MIKEY — WINDOWS CMD UI")
    print("=" * 60)

    print(
        "Health:",
        health.status_code,
        health.text
    )

    print(
        "UI:",
        page.status_code
    )

    print(
        "Port:",
        WEB_PORT
    )

    print("=" * 60)

    print(
        "✅ CMD UI READY."
    )

    print(
        "🌐 Cloudflare should point to port 5000."
    )

except Exception as e:

    print(
        "❌ Flask error:",
        e
    )

Starting Mikey CMD interface...
 * Serving Flask app 'mikey_cmd'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Sep/2026 18:23:03] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 18:23:03] "GET / HTTP/1.1" 200 -



MIKEY — WINDOWS CMD UI
Health: 200 {"service":"Mikey","status":"ok"}

UI: 200
Port: 5000
✅ CMD UI READY.
🌐 Cloudflare should point to port 5000.


In [19]:
# CELL 12 — FINAL HEALTH CHECK

import requests

print("=" * 60)
print("MIKEY FINAL CHECK")
print("=" * 60)

llama = requests.get(
    f"http://127.0.0.1:{LLAMA_PORT}/health",
    timeout=5,
)
web = requests.get(
    f"http://127.0.0.1:{WEB_PORT}/",
    timeout=5,
)

print("llama-server:", llama.status_code)
print("Web UI:", web.status_code)
print("Mikey UI:", "your private ai" in web.text)
print("Public URL:", public_url)
print("=" * 60)


127.0.0.1 - - [12/Sep/2026 18:23:18] "GET / HTTP/1.1" 200 -


MIKEY FINAL CHECK
llama-server: 200
Web UI: 200
Mikey UI: False
Public URL: https://meets-scope-casting-bases.trycloudflare.com


127.0.0.1 - - [12/Sep/2026 18:23:22] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 18:23:23] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [12/Sep/2026 18:24:32] "POST /chat HTTP/1.1" 200 -
127.0.0.1 - - [12/Sep/2026 18:27:19] "POST /chat HTTP/1.1" 200 -


In [20]:
# CELL 13 — STOP EVERYTHING
# Run only when you want to shut the chatbot down.
'''
for name in ["cloudflare_process", "llama_process"]:
    process = globals().get(name)

    if process is not None and process.poll() is None:
        process.terminate()
        print("Stopped:", name)

print("Done.")
'''

Stopped: cloudflare_process
Stopped: llama_process
Done.
